In [ ]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class SignLanguageNormalizer:
    """
    Normalizer specifically for 516-feature sign language data
    Feature structure: [POSE(132) | HANDS(126) | VELOCITY(258)]
    """
    
    def __init__(self, 
                 use_root_center=True,
                 use_shoulder_width=True,
                 use_hand_normalization=True,
                 use_scale_normalization=True,
                 use_clipping=True,
                 target_hand_size=0.3,
                 clip_bounds=(-2.0, 2.0)):
        """
        Initialize normalizer with configuration
        """
        self.use_root_center = use_root_center
        self.use_shoulder_width = use_shoulder_width
        self.use_hand_normalization = use_hand_normalization
        self.use_scale_normalization = use_scale_normalization
        self.use_clipping = use_clipping
        
        self.target_hand_size = target_hand_size
        self.clip_bounds = clip_bounds
        
        # Feature indices
        self.POSE_START = 0
        self.POSE_END = 132  # 33 landmarks × 4
        self.HANDS_START = 132
        self.HANDS_END = 258  # 126 features
        self.VELOCITY_START = 258
        self.VELOCITY_END = 516  # 258 features
        
        # MediaPipe landmark indices for pose
        self.LEFT_SHOULDER = 11
        self.RIGHT_SHOULDER = 12
        self.LEFT_WRIST = 15
        self.RIGHT_WRIST = 16
        self.LEFT_HIP = 23
        self.RIGHT_HIP = 24
        
        # Hand landmark indices (0-20)
        self.WRIST = 0
        self.FINGERTIPS = [4, 8, 12, 16, 20]  # Tips of all fingers
        
    def extract_pose_landmarks(self, features):
        """
        Extract pose landmarks from features
        Returns: (33, 4) array [x, y, z, visibility]
        """
        pose = features[self.POSE_START:self.POSE_END].reshape(33, 4)
        return pose
    
    def extract_hand_landmarks(self, features, hand='left'):
        """
        Extract hand landmarks from features
        Returns: (21, 3) array [x, y, z]
        """
        hand_features = features[self.HANDS_START:self.HANDS_END]
        
        if hand == 'left':
            hand_lms = hand_features[:63].reshape(21, 3)
        else:  # right
            hand_lms = hand_features[63:].reshape(21, 3)
        
        return hand_lms
    
    def extract_velocity(self, features):
        """
        Extract velocity features
        Returns: velocity array
        """
        velocity = features[self.VELOCITY_START:self.VELOCITY_END]
        return velocity
    
    def normalize_sequence(self, features_sequence):
        """
        Normalize a full sequence (30 frames) of 516-feature vectors
        
        Args:
            features_sequence: (T, 516) array
        
        Returns:
            normalized_sequence: (T, 516) array with normalized features
        """
        T = features_sequence.shape[0]
        normalized_features = []  # Use list instead of numpy array
        
        # First pass: collect shoulder widths for all frames
        shoulder_widths = []
        for t in range(T):
            features = features_sequence[t]
            pose = self.extract_pose_landmarks(features)
            
            if self.use_shoulder_width:
                left_shoulder = pose[self.LEFT_SHOULDER][:3]  # x, y, z
                right_shoulder = pose[self.RIGHT_SHOULDER][:3]
                width = np.linalg.norm(left_shoulder - right_shoulder)
                shoulder_widths.append(width)
            else:
                shoulder_widths.append(1.0)
        
        shoulder_widths = np.array(shoulder_widths)
        
        # Second pass: normalize each frame
        for t in range(T):
            features = features_sequence[t].copy()
            
            # Extract components
            pose = self.extract_pose_landmarks(features)
            left_hand = self.extract_hand_landmarks(features, 'left')
            right_hand = self.extract_hand_landmarks(features, 'right')
            velocity = self.extract_velocity(features)
            
            # ──────────────────────────────────────────────
            # 1. NORMALIZE POSE LANDMARKS
            # ──────────────────────────────────────────────
            if self.use_root_center:
                # Use hip center as root for pose
                left_hip = pose[self.LEFT_HIP][:3]
                right_hip = pose[self.RIGHT_HIP][:3]
                hip_center = (left_hip + right_hip) / 2
                
                # Subtract hip center from all pose landmarks
                pose[:, :3] = pose[:, :3] - hip_center
            
            if self.use_shoulder_width:
                width = shoulder_widths[t]
                if width > 0.001:
                    pose[:, :3] = pose[:, :3] / width
                else:
                    pose[:, :3] = 0
            
            # ──────────────────────────────────────────────
            # 2. NORMALIZE HAND LANDMARKS
            # ──────────────────────────────────────────────
            
            # Normalize left hand
            if self.use_root_center:
                # Subtract wrist from left hand
                left_wrist = left_hand[self.WRIST]
                left_hand = left_hand - left_wrist
            
            if self.use_scale_normalization:
                left_hand = self._normalize_hand_scale(left_hand)
            
            # Normalize right hand
            if self.use_root_center:
                right_wrist = right_hand[self.WRIST]
                right_hand = right_hand - right_wrist
            
            if self.use_scale_normalization:
                right_hand = self._normalize_hand_scale(right_hand)
            
            # ──────────────────────────────────────────────
            # 3. LEFT/RIGHT HAND NORMALIZATION
            # ──────────────────────────────────────────────
            if self.use_hand_normalization:
                # Mirror left hand to match right hand convention
                left_hand[:, 0] = -left_hand[:, 0]
            
            # ──────────────────────────────────────────────
            # 4. CLIP OUTLIERS
            # ──────────────────────────────────────────────
            if self.use_clipping:
                pose[:, :3] = np.clip(pose[:, :3], 
                                     self.clip_bounds[0], 
                                     self.clip_bounds[1])
                left_hand = np.clip(left_hand, 
                                   self.clip_bounds[0], 
                                   self.clip_bounds[1])
                right_hand = np.clip(right_hand, 
                                    self.clip_bounds[0], 
                                    self.clip_bounds[1])
            
            # ──────────────────────────────────────────────
            # 5. COMBINE FEATURES (without velocity)
            # ──────────────────────────────────────────────
            pose_flat = pose.flatten()
            hands_flat = np.concatenate([left_hand.flatten(), right_hand.flatten()])
            combined = np.concatenate([pose_flat, hands_flat])
            
            # For now, keep original velocity (will be recomputed later)
            combined = np.concatenate([combined, velocity])
            
            normalized_features.append(combined)
        
        normalized_features = np.array(normalized_features)
        
        # ──────────────────────────────────────────────
        # 6. RECOMPUTE VELOCITY
        # ──────────────────────────────────────────────
        # Extract normalized pose + hands (without velocity)
        pose_hands = normalized_features[:, :258]  # First 258 features (pose+hands)
        velocity = self._compute_velocity(pose_hands)
        
        # Combine with recomputed velocity
        normalized_features = np.concatenate([pose_hands, velocity], axis=1)
        
        return normalized_features
    
    def _normalize_hand_scale(self, hand_landmarks):
        """
        Normalize hand to target size
        """
        wrist = hand_landmarks[self.WRIST]
        
        # Calculate hand size from fingertips
        distances = []
        for idx in self.FINGERTIPS:
            if idx < len(hand_landmarks):
                dist = np.linalg.norm(hand_landmarks[idx] - wrist)
                distances.append(dist)
        
        hand_size = np.mean(distances) if distances else 1.0
        
        if hand_size > 0.001:
            scale_factor = self.target_hand_size / hand_size
            hand_landmarks = hand_landmarks * scale_factor
        
        return hand_landmarks
    
    def _compute_velocity(self, pose_hands):
        """
        Compute velocity from normalized pose+hands
        """
        T = pose_hands.shape[0]
        velocity = np.zeros_like(pose_hands)
        
        # Compute velocity as difference between consecutive frames
        velocity[1:] = pose_hands[1:] - pose_hands[:-1]
        # First frame velocity is zero
        
        return velocity
    
    def process_file(self, input_path, output_path):
        """
        Process a single .npy file
        """
        data = np.load(input_path)
        
        # Ensure data has correct shape
        if data.ndim == 1:
            data = data.reshape(1, -1)
        
        # Normalize
        normalized = self.normalize_sequence(data)
        
        # Save
        np.save(output_path, normalized)
        
        return normalized
    
    def process_directory(self, input_dir, output_dir, classes=None, max_samples=None):
        """
        Process all .npy files in a directory
        """
        input_dir = Path(input_dir)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Get class folders
        if classes is None:
            class_folders = [d for d in input_dir.iterdir() if d.is_dir()]
        else:
            class_folders = [input_dir / cls for cls in classes if (input_dir / cls).exists()]
        
        print(f"Processing {len(class_folders)} classes...")
        
        total_files = 0
        total_frames = 0
        
        for class_path in tqdm(class_folders, desc="Processing classes"):
            class_name = class_path.name
            out_class_path = output_dir / class_name
            out_class_path.mkdir(parents=True, exist_ok=True)
            
            npy_files = sorted(class_path.glob("*.npy"))
            
            if max_samples:
                npy_files = npy_files[:max_samples]
            
            for file_path in tqdm(npy_files, desc=f"  {class_name}", leave=False):
                try:
                    output_path = out_class_path / file_path.name
                    normalized = self.process_file(file_path, output_path)
                    total_files += 1
                    total_frames += len(normalized)
                except Exception as e:
                    print(f"   Error processing {file_path.name}: {e}")
        
        print(f"\n Processing complete!")
        print(f"  - Files processed: {total_files}")
        print(f"  - Total frames: {total_frames}")

# ============================================================
# TEST THE NORMALIZER
# ============================================================

def test_normalizer():
    """Test the normalizer with sample data"""
    
    print("=" * 60)
    print("TESTING 516-FEATURE NORMALIZER")
    print("=" * 60)
    
    # Create sample 516-feature data
    # 30 frames × 516 features
    sample_data = np.random.randn(30, 516) * 0.1
    
    # Add some structure to make it realistic
    for t in range(30):
        # Pose landmarks (33 × 4)
        pose = sample_data[t, :132].reshape(33, 4)
        
        # Add shoulder landmarks (indices 11, 12)
        pose[11, :3] = np.array([-0.2, 0.1, 0])  # Left shoulder
        pose[12, :3] = np.array([0.2, 0.1, 0])   # Right shoulder
        
        # Add wrist landmarks (indices 15, 16)
        pose[15, :3] = np.array([-0.4, 0.2, 0])  # Left wrist
        pose[16, :3] = np.array([0.4, 0.2, 0])   # Right wrist
        
        # Add hip landmarks (indices 23, 24)
        pose[23, :3] = np.array([-0.1, -0.2, 0])
        pose[24, :3] = np.array([0.1, -0.2, 0])
        
        # Update pose
        sample_data[t, :132] = pose.flatten()
        
        # Hand landmarks (21 × 3 for left hand)
        left_hand = sample_data[t, 132:195].reshape(21, 3)
        # Put wrist at origin
        left_hand[0] = np.array([0, 0, 0])
        # Add fingertips
        for idx in [4, 8, 12, 16, 20]:
            left_hand[idx] = np.random.randn(3) * 0.1 + np.array([0, 0.2, 0])
        sample_data[t, 132:195] = left_hand.flatten()
        
        # Right hand
        right_hand = sample_data[t, 195:258].reshape(21, 3)
        right_hand[0] = np.array([0, 0, 0])
        for idx in [4, 8, 12, 16, 20]:
            right_hand[idx] = np.random.randn(3) * 0.1 + np.array([0, 0.2, 0])
        sample_data[t, 195:258] = right_hand.flatten()
        
        # Velocity (zeros for simplicity)
        sample_data[t, 258:] = 0
    
    # Normalize
    normalizer = SignLanguageNormalizer(
        use_root_center=True,
        use_shoulder_width=True,
        use_hand_normalization=True,
        use_scale_normalization=True,
        use_clipping=True
    )
    
    normalized = normalizer.normalize_sequence(sample_data)
    
    print(f"\n Before Normalization:")
    print(f"  - Shape: {sample_data.shape}")
    print(f"  - Mean: {sample_data.mean():.3f}")
    print(f"  - Std: {sample_data.std():.3f}")
    print(f"  - Min: {sample_data.min():.3f}")
    print(f"  - Max: {sample_data.max():.3f}")
    
    print(f"\n After Normalization:")
    print(f"  - Shape: {normalized.shape}")
    print(f"  - Mean: {normalized.mean():.3f}")
    print(f"  - Std: {normalized.std():.3f}")
    print(f"  - Min: {normalized.min():.3f}")
    print(f"  - Max: {normalized.max():.3f}")
    
    # Verify normalization worked
    print(f"\n Normalization Summary:")
    if normalized.std() < sample_data.std():
        print(f"  - Reduced variance: {sample_data.std():.3f} → {normalized.std():.3f}")
    if abs(normalized.mean()) < 0.1:
        print(f"  - Centered around zero: {normalized.mean():.3f}")
    if normalized.max() < 3.0 and normalized.min() > -3.0:
        print(f"  - Clipped to bounds: [{normalized.min():.3f}, {normalized.max():.3f}]")
    
    return normalized

# ============================================================
# QUICK TEST ON YOUR DATA
# ============================================================

def quick_test_on_real_data(input_file):
    """
    Quick test on a real data file
    """
    print("\n" + "=" * 60)
    print(f"TESTING ON REAL DATA: {input_file}")
    print("=" * 60)
    
    # Load data
    data = np.load(input_file)
    print(f"\nData shape: {data.shape}")
    
    if data.ndim == 1:
        data = data.reshape(1, -1)
    
    print(f"Frames: {data.shape[0]}")
    print(f"Features: {data.shape[1]}")
    
    # Check if it's 516 features
    if data.shape[1] == 516:
        print(" Correct feature size (516)")
    else:
        print(f" Expected 516 features, got {data.shape[1]}")
    
    # Check for zero frames
    zero_frames = 0
    for t in range(data.shape[0]):
        if np.allclose(data[t], 0):
            zero_frames += 1
    
    print(f"Zero frames: {zero_frames}/{data.shape[0]}")
    
    # Normalize
    print("\nNormalizing...")
    normalizer = SignLanguageNormalizer(
        use_root_center=True,
        use_shoulder_width=True,
        use_hand_normalization=True,
        use_scale_normalization=True,
        use_clipping=True
    )
    
    normalized = normalizer.normalize_sequence(data)
    
    print(f"\n After Normalization:")
    print(f"  - Shape: {normalized.shape}")
    print(f"  - Mean: {normalized.mean():.6f}")
    print(f"  - Std: {normalized.std():.6f}")
    print(f"  - Min: {normalized.min():.6f}")
    print(f"  - Max: {normalized.max():.6f}")
    
    print("\n Normalization test complete!")

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    # First, test with sample data
    test_normalizer()
    
    # Optionally, test on real data
    print("\n" + "=" * 60)
    print("TEST ON REAL DATA?")
    print("=" * 60)
    
    # Example: test on one of your files
    test_file = Path(r"D:\uni\Intern-1-Project\ksl\landmarks_30frames_smart\កៅអី\000000.npy")
    
    if test_file.exists():
        quick_test_on_real_data(test_file)
    else:
        print(f"\n Test file not found: {test_file}")
        print("Update the path to your actual data file.")
    
    print("\n" + "=" * 60)
    print("USAGE EXAMPLE:")
    print("=" * 60)

    # Normalize your extracted data
    normalizer = SignLanguageNormalizer(
    use_root_center=True,
    use_shoulder_width=True,
    use_hand_normalization=True,
    use_scale_normalization=True,
    use_clipping=True,
    target_hand_size=0.3,
    clip_bounds=(-2.0, 2.0)
)
    
    # Process all files
    normalizer.process_directory(
        input_dir="D:/uni/Intern-1-Project/ksl/landmarks_30frames_smart_old",
        output_dir="D:/uni/Intern-1-Project/ksl/landmarks_30frames_normalized_v2",
        classes=['កុំព្យូទ័រ','កៅអី','ក្ដារខៀន','ខ្មៅដៃ','ជ័រលុប','ដីស','តុ','ទឹកលុប','នាយករង','នាយិកា','បន្ទាត់','សៀវភៅ','ប៊ិកខៀវ','ហ្វឺតខ្មៅ',
        'ហ្វឺតក្រហម','ប៊ិក','ប៊ិកក្រហម','កាតាប','កាតាបស្ពាយក្រោយ','ហ្វឺតខៀវ'],
        max_samples=None  # Optional: limit samples
    )